# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# set dl url for January 2019 trip data
# download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
# response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [2]:
# create the dataframe
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [35]:
# Show the dataframe
df_trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|  duration_minutes|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [17]:
# Add a column that creates a unique key to identify each record in order to answer questions about individual trips
from pyspark.sql.functions import monotonically_increasing_id

df_trips = df_trips.withColumn("trip_id", monotonically_increasing_id())
df_trips.select("trip_id", "tpep_pickup_datetime", "passenger_count").show(10)

+-----------+--------------------+---------------+
|    trip_id|tpep_pickup_datetime|passenger_count|
+-----------+--------------------+---------------+
|25769803776| 2019-01-01 00:46:40|            1.0|
|25769803777| 2019-01-01 00:59:47|            1.0|
|25769803778| 2018-12-21 13:48:30|            3.0|
|25769803779| 2018-11-28 15:52:25|            5.0|
|25769803780| 2018-11-28 15:56:57|            5.0|
|25769803781| 2018-11-28 16:25:49|            5.0|
|25769803782| 2018-11-28 16:29:37|            5.0|
|25769803783| 2019-01-01 00:21:28|            1.0|
|25769803784| 2019-01-01 00:32:01|            1.0|
|25769803785| 2019-01-01 00:57:32|            2.0|
+-----------+--------------------+---------------+
only showing top 10 rows


In [18]:
# Which trip has the highest passanger count
from pyspark.sql.functions import col

df_trips.orderBy(col("passenger_count").desc()).select("trip_id", "tpep_pickup_datetime", "passenger_count").show(1)

+-----------+--------------------+---------------+
|    trip_id|tpep_pickup_datetime|passenger_count|
+-----------+--------------------+---------------+
|25770753732| 2019-01-05 13:12:29|            9.0|
+-----------+--------------------+---------------+
only showing top 1 row


In [19]:
# What is the Average passenger count
from pyspark.sql.functions import avg

df_trips.select(avg("passenger_count")).show()

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [25]:
# Shortest/longest trip by distance?
df_trips.orderBy(col("trip_distance").desc()).select("trip_id", "tpep_pickup_datetime", "trip_distance").show(1)
df_trips.orderBy(col("trip_distance").asc()).select("trip_id", "tpep_pickup_datetime", "trip_distance").show(1)

# Taking the first shortest trip by distance greater than 0
df_trips.filter(col("trip_distance") > 0).orderBy(col("trip_distance").asc()).select("trip_id", "tpep_pickup_datetime", "trip_distance").show(1)

+-----------+--------------------+-------------+
|    trip_id|tpep_pickup_datetime|trip_distance|
+-----------+--------------------+-------------+
|25775877867| 2019-01-25 21:56:39|        831.8|
+-----------+--------------------+-------------+
only showing top 1 row
+-----------+--------------------+-------------+
|    trip_id|tpep_pickup_datetime|trip_distance|
+-----------+--------------------+-------------+
|25769803778| 2018-12-21 13:48:30|          0.0|
+-----------+--------------------+-------------+
only showing top 1 row
+-----------+--------------------+-------------+
|    trip_id|tpep_pickup_datetime|trip_distance|
+-----------+--------------------+-------------+
|25769816340| 2019-01-01 00:02:54|         0.01|
+-----------+--------------------+-------------+
only showing top 1 row


In [26]:
# Shortest/longest trip by time
from pyspark.sql.functions import unix_timestamp

df_trips = df_trips.withColumn("duration_minutes",(unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60)
df_trips.orderBy(col("duration_minutes").desc()).select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_minutes").show(1)
df_trips.orderBy(col("duration_minutes").asc()).select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_minutes").show(1)

# Taking the first shortest trip by time greater than 0 minute
df_trips.filter(col("duration_minutes") > 0).orderBy(col("duration_minutes").asc()).select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_minutes").show(1)

+-----------+--------------------+---------------------+-----------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime| duration_minutes|
+-----------+--------------------+---------------------+-----------------+
|25769872043| 2019-01-01 07:01:20|  2019-01-31 14:29:21|43648.01666666667|
+-----------+--------------------+---------------------+-----------------+
only showing top 1 row
+-----------+--------------------+---------------------+----------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|duration_minutes|
+-----------+--------------------+---------------------+----------------+
|25771006960| 2019-01-06 15:15:08|  2018-11-09 02:34:38|        -84280.5|
+-----------+--------------------+---------------------+----------------+
only showing top 1 row
+-----------+--------------------+---------------------+--------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|    duration_minutes|
+-----------+--------------------+-------------------

In [31]:
# Busiest day/slowest single day
from pyspark.sql.functions import to_date

trips_by_date = df_trips.groupBy(to_date("tpep_pickup_datetime").alias("date")).agg(count("*").alias("trip_count"))
trips_by_date.orderBy(col("trip_count").desc()).show(1)
trips_by_date.orderBy(col("trip_count").asc()).show(1)

+----------+----------+
|      date|trip_count|
+----------+----------+
|2019-01-25|    292499|
+----------+----------+
only showing top 1 row
+----------+----------+
|      date|trip_count|
+----------+----------+
|2019-05-20|         1|
+----------+----------+
only showing top 1 row


In [32]:
# Busiest day/slowest single time of the day
from pyspark.sql.functions import hour

trips_by_hour = df_trips.groupBy(hour("tpep_pickup_datetime").alias("hour_of_the_day")).agg(count("*").alias("trip_count"))
trips_by_hour.orderBy(col("trip_count").desc()).show(1)
trips_by_hour.orderBy(col("trip_count").asc()).show(1)

+---------------+----------+
|hour_of_the_day|trip_count|
+---------------+----------+
|             18|    515390|
+---------------+----------+
only showing top 1 row
+---------------+----------+
|hour_of_the_day|trip_count|
+---------------+----------+
|              4|     61424|
+---------------+----------+
only showing top 1 row


In [34]:
# On average which day of the week is slowest/busiest
from pyspark.sql.functions import date_format

trips_by_day = df_trips.groupBy(date_format("tpep_pickup_datetime", "E").alias("day_of_the_week")).agg(count("*").alias("trip_count"))
trips_by_day.orderBy(col("trip_count").desc()).show(1)
trips_by_day.orderBy(col("trip_count").asc()).show(1)

+---------------+----------+
|day_of_the_week|trip_count|
+---------------+----------+
|            Thu|   1357043|
+---------------+----------+
only showing top 1 row
+---------------+----------+
|day_of_the_week|trip_count|
+---------------+----------+
|            Sun|    859905|
+---------------+----------+
only showing top 1 row


In [42]:
# Does trip distance or num passangers affect tip amount
from pyspark.sql.functions import corr

df_trips.select(corr("trip_distance", "tip_amount").alias("corr_distance_tip"),corr("passenger_count", "tip_amount").alias("corr_passengers_tip")).show()

# Distance maybe, Passenger_count no

+------------------+--------------------+
| corr_distance_tip| corr_passengers_tip|
+------------------+--------------------+
|0.5269200663652668|0.001084223312167...|
+------------------+--------------------+



In [44]:
# What was the highest "extra" charge and which trip

df_trips.orderBy(col("extra").desc()).select("trip_id", "tpep_pickup_datetime", "passenger_count", "extra", "tip_amount").show(1)

+-----------+--------------------+---------------+------+----------+
|    trip_id|tpep_pickup_datetime|passenger_count| extra|tip_amount|
+-----------+--------------------+---------------+------+----------+
|25775127259| 2019-01-23 08:58:09|            1.0|535.38|       0.0|
+-----------+--------------------+---------------+------+----------+
only showing top 1 row


# Are there any datapoints that seem to be strange/outliers
Dropoff time earlier than pickup time
Fares with duration of 0 minute and of a few seconds
Trip distance of 0
0 passenger counts
Dataset from 2019 but containing trip from 2028

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [45]:
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing